# RACE RC & Quiz Generation - ULTIMATE MASTER NOTEBOOK (v4 - Model B Fix)
This version ensures Model B uses LogisticRegression for hints to match the rubric and code logic.

In [ ]:
# 1. Setup & Installation
!pip install gensim joblib pandas numpy scikit-learn

import os, joblib, time, re
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Paths
PROJECT_DIR = '/content/drive/MyDrive/race_rc_project'
DATA_DIR    = f'{PROJECT_DIR}/data/processed'
MODEL_DIR_A = f'{PROJECT_DIR}/models/model_a/traditional'
MODEL_DIR_B = f'{PROJECT_DIR}/models/model_b/traditional'
os.makedirs(MODEL_DIR_A, exist_ok=True)
os.makedirs(MODEL_DIR_B, exist_ok=True)

# Load Data
print("Loading features...")
train_data = joblib.load(f'{DATA_DIR}/train_features.pkl')
val_data   = joblib.load(f'{DATA_DIR}/val_features.pkl')

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val     = val_data['X'], val_data['y']
print(f"Loaded {len(X_train)} training rows.")

## 2. Preprocessing: Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
print("Fitting Scaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
joblib.dump(scaler, f'{MODEL_DIR_A}/scaler.pkl')
print("Scaler saved.")

## 3. Model A: Supervised & Ensemble

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

print("Training Supervised Models...")
# 3a. LR
lr = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr, f'{MODEL_DIR_A}/lr.pkl')

# 3b. SVM
svm = LinearSVC(class_weight='balanced', C=0.5, max_iter=2000, dual='auto')
svm.fit(X_train_scaled, y_train)
joblib.dump(svm, f'{MODEL_DIR_A}/svm.pkl')

# 3c. Naive Bayes
nb = BernoulliNB()
nb.fit((X_train_scaled > 0).astype(float), y_train)
joblib.dump(nb, f'{MODEL_DIR_A}/nb.pkl')

# 3d. Ensemble
print("Training Ensemble...")
svm_cal = CalibratedClassifierCV(svm, cv=3)
ensemble = VotingClassifier(estimators=[('lr', lr), ('svm', svm_cal), ('nb', nb)], voting='soft')
ensemble.fit(X_train_scaled[:20000], y_train[:20000])
joblib.dump(ensemble, f'{MODEL_DIR_A}/ensemble.pkl')

## 4. Unsupervised & Semi-Supervised

In [ ]:
from sklearn.cluster import KMeans
from sklearn.semi_supervised import LabelPropagation

print("Running KMeans...")
km = KMeans(n_clusters=4, n_init='auto')
km.fit(X_train_scaled[:10000])
joblib.dump(km, f'{MODEL_DIR_A}/kmeans.pkl')

print("Running Label Propagation...")
lp = LabelPropagation(max_iter=500)
lp.fit(X_train_scaled[:5000], y_train[:5000])
joblib.dump(lp, f'{MODEL_DIR_A}/label_prop.pkl')

## 5. Model B: Distractors & Hints

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLR

print("Training Model B Rankers (LogisticRegression for both)...")
# Distractor Ranker
dist_lr = SkLR(class_weight='balanced')
dist_lr.fit(X_train_scaled[:10000], y_train[:10000])
joblib.dump(dist_lr, f'{MODEL_DIR_B}/distractor_lr.pkl')

# Hint Scorer (Must be LogisticRegression to support .predict_proba)
hint_lr = SkLR(class_weight='balanced')
hint_lr.fit(X_train_scaled[:10000], y_train[:10000])
joblib.dump(hint_lr, f'{MODEL_DIR_B}/hint_lr.pkl')

import gensim.downloader as api
print("Downloading Word2Vec...")
w2v = api.load('word2vec-google-news-300')
w2v.save(f'{MODEL_DIR_B}/w2v.kv')
print("ALL DONE! MODELS COMPATIBLE AND SCALED.")